# Explorando Self-Attention — el corazón del Transformer

Vamos a implementar self-attention paso a paso, igual que hicimos con BPE.
Después formalizamos en `src/transformer/self_attention.py`.

**Plan:**
1. Setup: tensor de ejemplo (4 tokens, 8-dim embeddings)
2. Crear las matrices W_q, W_k, W_v (learnable)
3. Computar Q, K, V
4. Attention scores: Q @ K^T
5. Scaling + softmax
6. Weighted sum: attn @ V
7. Output: las mismas 4 posiciones, ahora contextualizadas

In [1]:
import torch

# Simulemos 4 tokens ya embedidos (en la práctica vienen del embedding layer)
# Cada token es un vector de 8 dimensiones
x = torch.tensor([
    [1.0, 0.5, 0.2, 0.8, 0.1, 0.4, 0.3, 0.9],   # token 1
    [0.2, 0.9, 0.4, 0.1, 0.7, 0.3, 0.8, 0.5],   # token 2
    [0.6, 0.1, 0.9, 0.4, 0.2, 0.7, 0.5, 0.3],   # token 3
    [0.3, 0.7, 0.2, 0.9, 0.5, 0.6, 0.1, 0.8],   # token 4
])

print(f"Forma del input: {x.shape}")  # torch.Size([4, 8])
print(f"  → 4 tokens, cada uno con embedding de 8 dimensiones")

Forma del input: torch.Size([4, 8])
  → 4 tokens, cada uno con embedding de 8 dimensiones


In [ ]:
import torch.nn as nn

d_model = 8 # dimensión del embedding

# Tres proyecciones learnable (matrices de pesos)
# bias=False para simplificar — la versión original usa bias pero no es esencial
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

# Aplicar las proyecciones a x
# Cada token se vuelve 3 "vistas" de sí mismo: Q, K, V
Q = W_q(x)  # query: lo que cada token "pregunta"
K = W_k(x)  # key: lo que cada token "ofrece"
V = W_v(x)  # value: el contenido de cada token

print(f"Q shape: {Q.shape}")  # torch.Size([4, 8])
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

Q shape: torch.Size([4, 8])
K shape: torch.Size([4, 8])
V shape: torch.Size([4, 8])


In [3]:
import torch.nn.functional as F
import math

# Attention scores: Q @ K^T
# Q: (4, 8), K.transpose: (8, 4) → scores: (4, 4)
scores = Q @ K.T

# Scaling: dividimos por sqrt(d_k) para estabilizar el softmax
# (sin esto, con d_model grande los scores explotan y el softmax da casi one-hot)
d_k = K.shape[-1]  # 8
scores = scores / math.sqrt(d_k)


print(f"Scores shape: {scores.shape}")  # torch.Size([4, 4])
print(f"Scores matrix (raw):")
print(scores)

# Softmax: convierte cada fila en probabilidades (suman 1)
attn = F.softmax(scores, dim=-1)

print(f"\nAttention weights (después de softmax):")
print(attn)
print(f"\nSuma de cada fila (debe ser 1.0): {attn.sum(dim=-1)}")

Scores shape: torch.Size([4, 4])
Scores matrix (raw):
tensor([[-0.1410, -0.0631, -0.1012, -0.1895],
        [ 0.0544,  0.0760,  0.0336,  0.0439],
        [-0.0426,  0.0354, -0.0285, -0.0605],
        [-0.0271, -0.0808, -0.0116, -0.1282]], grad_fn=<DivBackward0>)

Attention weights (después de softmax):
tensor([[0.2454, 0.2653, 0.2554, 0.2338],
        [0.2506, 0.2560, 0.2454, 0.2480],
        [0.2452, 0.2651, 0.2487, 0.2409],
        [0.2586, 0.2451, 0.2626, 0.2337]], grad_fn=<SoftmaxBackward0>)

Suma de cada fila (debe ser 1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [4]:
output = attn @ V

print(f"Output shape: {output.shape}")  # torch.Size([4, 8])
print(f"\nOutput[0] (representación contextualizada del token 1):")
print(output[0])

Output shape: torch.Size([4, 8])

Output[0] (representación contextualizada del token 1):
tensor([-0.1749, -0.0629, -0.0978,  0.0556,  0.3055, -0.2076, -0.1771, -0.5248],
       grad_fn=<SelectBackward0>)
